# 03 · Feature extraction — lead III (D3)

Segments each record into R-R intervals (R-peaks from V4, first minute) and computes 10 statistical and entropy features on lead III → `fd3.csv`.

> Originally run on Google Colab; saved outputs are kept. Set the paths below to rerun locally.

In [ ]:
# Paths — put the PhysioNet zip files in DATA_DIR (see ../data/README.md)
import os
import pandas as pd

DATA_DIR = "../data/raw"        # ptb-diagnostic-ecg-database-1.0.0.zip, european-st-t-database-1.0.0.zip
WORK_DIR = "../data/work"       # extracted records and intermediate CSVs
LABELS_CSV = "../data/labels.csv"
os.makedirs(WORK_DIR, exist_ok=True)


def load_patient_labels():
    """Diagnosis of each ischemic patient (replaces the original data2.csv)."""
    df = pd.read_csv(LABELS_CSV)
    df = df[df.label > 0]
    return pd.DataFrame({"ID patient ": df.record, "label": df.label_name})


In [ ]:
import os
import zipfile
import shutil

# Input zip paths
data_dir = DATA_DIR
ptb_zip_path = os.path.join(data_dir, 'ptb-diagnostic-ecg-database-1.0.0.zip')
european_zip_path = os.path.join(data_dir, 'european-st-t-database-1.0.0.zip')

# Output structure
output_base = f"{WORK_DIR}/ECG_Data"
ischemic_dir = os.path.join(output_base, 'Ischemic')
normal_dir = os.path.join(output_base, 'Normal')
os.makedirs(ischemic_dir, exist_ok=True)
os.makedirs(normal_dir, exist_ok=True)

# Temporary extract folders
ptb_temp = f"{WORK_DIR}/ptb-temp"
european_temp = f"{WORK_DIR}/european-temp"

# --- Unzip PTB ---
with zipfile.ZipFile(ptb_zip_path, 'r') as zip_ref:
    zip_ref.extractall(ptb_temp)

ptb_root = os.path.join(ptb_temp, 'ptb-diagnostic-ecg-database-1.0.0')

# Flatten PTB: Copy and rename files
for patient_folder in os.listdir(ptb_root):
    full_path = os.path.join(ptb_root, patient_folder)
    if os.path.isdir(full_path):
        for fname in os.listdir(full_path):
            src = os.path.join(full_path, fname)
            base_name = f"{fname}"  # Avoid filename collisions
            dst = os.path.join(normal_dir, base_name)
            shutil.copyfile(src, dst)

# --- Unzip European ST-T ---
with zipfile.ZipFile(european_zip_path, 'r') as zip_ref:
    zip_ref.extractall(european_temp)

european_root = os.path.join(european_temp, 'european-st-t-database-1.0.0')

# Copy all files from European ST-T
for fname in os.listdir(european_root):
    src = os.path.join(european_root, fname)
    dst = os.path.join(ischemic_dir, fname)
    shutil.copyfile(src, dst)


In [ ]:
!pip install wfdb

In [ ]:
import numpy as np
import wfdb
from scipy.stats import kurtosis, skew
from scipy.signal import welch
from math import log


In [ ]:
from scipy.signal import butter, filtfilt, find_peaks
import numpy as np

def pan_tompkins(ecg_signal, fs):
    """
    Implements Pan-Tompkins QRS detection algorithm.
    Returns:
        r_peaks: Indices of detected R-peaks
        mwa: Moving window integrated signal
    """



    # === 2. Derivative
    diff = np.diff(ecg_signal, prepend=ecg_signal[0])

    # === 3. Squaring
    squared = diff ** 2

    # === 4. Moving window integration (~150ms)
    window_size = int(0.15 * fs)
    mwa = np.convolve(squared, np.ones(window_size)/window_size, mode='same')

    # === 5. Peak detection on MWI signal
    threshold = np.mean(mwa) * 1.5
    raw_peaks, _ = find_peaks(mwa, height=threshold, distance=int(0.6 * fs))

    # === 6. Refine R-peaks: search for local max around each peak in original signal
    search_radius = int(0.1 * fs)  # 100ms before and after
    r_peaks = []

    for peak in raw_peaks:
        start = max(0, peak - search_radius)
        end = min(len(ecg_signal), peak + search_radius)
        local_max_idx = np.argmax(ecg_signal[start:end])
        r_peaks.append(start + local_max_idx)

    return np.array(r_peaks), mwa




# === PQRST Detection Function ===
def detect_pqrst(signal, r_peaks, fs):
    p_waves = []
    q_waves = []
    s_waves = []
    t_waves = []

    for r in r_peaks:
        # Define windows (you can tune these values)
        p_win = int(0.2 * fs)
        q_win = int(0.05 * fs)
        s_win = int(0.05 * fs)
        t_win = int(0.4 * fs)

        # Q wave = min before R
        if r - q_win >= 0:
            q_segment = signal[r - q_win:r]
            q_idx = np.argmin(q_segment)
            q_waves.append(r - q_win + q_idx)

        # S wave = min after R
        if r + s_win < len(signal):
            s_segment = signal[r:r + s_win]
            s_idx = np.argmin(s_segment)
            s_waves.append(r + s_idx)

        # P wave = max before Q (or R if Q not found)
        p_start = max(0, r - p_win)
        p_segment = signal[p_start:r - q_win] if r - q_win > p_start else signal[p_start:r]
        if len(p_segment) > 0:
            p_idx = np.argmax(p_segment)
            p_waves.append(p_start + p_idx)

        # T wave = max after S
        t_start = r + s_win
        t_end = min(len(signal), r + t_win)
        t_segment = signal[t_start:t_end]
        if len(t_segment) > 0:
            t_idx = np.argmax(t_segment)
            t_waves.append(t_start + t_idx)

    return np.array(p_waves), np.array(q_waves), np.array(s_waves), np.array(t_waves)

In [ ]:
def extract_stat_entropy_features(segment):
    N = len(segment)
    if N == 0:
        return [0]*10  # Handle empty segments gracefully

    A_max = np.max(segment)
    A_min = np.min(segment)
    mean = np.mean(segment)
    std = np.std(segment)
    rms = np.sqrt(np.mean(segment**2))
    kurt = kurtosis(segment)
    skewness = skew(segment)
    sample_entropy = sampen(segment, m=2, r=0.2 * std)
    perm_entropy = permutation_entropy(segment, m=3)
    approx_entropy = approximate_entropy(segment, m=2, r=0.2 * std)

    return [A_max, A_min, mean, std, rms, kurt, skewness, sample_entropy, perm_entropy, approx_entropy]


In [ ]:
def sampen(series, m, r):
    N = len(series)
    if N < m + 1:
        return 0
    def _phi(m):
        x = np.array([series[i:i + m] for i in range(N - m + 1)])
        C = np.sum(np.max(np.abs(x[:, None] - x[None, :]), axis=2) < r, axis=0) - 1
        return np.sum(C) / (N - m + 1)
    try:
        return -np.log(_phi(m + 1) / _phi(m))
    except:
        return 0

def permutation_entropy(time_series, m):
    from math import factorial
    N = len(time_series)
    if N < m:
        return 0
    perms = {}
    for i in range(N - m + 1):
        key = tuple(np.argsort(time_series[i:i + m]))
        perms[key] = perms.get(key, 0) + 1
    probs = np.array(list(perms.values()), dtype=float)
    probs /= probs.sum()
    return -np.sum(probs * np.log(probs)) / log(factorial(m))

def approximate_entropy(U, m, r):
    N = len(U)
    if N <= m + 1:
        return 0
    def _phi(m):
        x = np.array([U[i:i + m] for i in range(N - m + 1)])
        C = np.sum(np.max(np.abs(x[:, None] - x[None, :]), axis=2) <= r, axis=0) / (N - m + 1)
        return np.sum(np.log(C)) / (N - m + 1)
    try:
        return _phi(m) - _phi(m + 1)
    except:
        return 0


In [ ]:
import numpy as np
from scipy.signal import butter, filtfilt, iirnotch
import cupy as cp

# === High-pass filter ===
def highpass_filter(signal, cutoff, fs, order=4):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='high', analog=False)
    return filtfilt(b, a, signal)

# === Low-pass filter ===
def lowpass_filter(signal, cutoff, fs, order=4):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return filtfilt(b, a, signal)

# === Notch filter ===
def notch_filter(signal, notch_freq, fs, quality=30):
    nyq = 0.5 * fs
    b, a = iirnotch(notch_freq / nyq, quality)
    return filtfilt(b, a, signal)

# === Apply all filters ===
def preprocess_ecg(signal, fs):
    # Convert CuPy input to NumPy for SciPy filtering
    if isinstance(signal, cp.ndarray):
        signal = signal.get()

    # 1. High-pass filter
    signal = highpass_filter(signal, 0.5, fs)

    # 2. Low-pass filter

    # 3. Notch filter (50 Hz for EU/UK, 60 Hz for US)
    signal = notch_filter(signal, 50.0, fs)
    signal = lowpass_filter(signal, 100.0, fs)

    return signal  # Still NumPy — convert to CuPy later if needed


In [ ]:
import pandas as pd

In [ ]:
# Load label info
labels_df = load_patient_labels()
labels_df.columns = ['filename', 'label']
labels_map = {row['filename']: row['label'] for _, row in labels_df.iterrows()}
labels_map

{'e0103': 'Angine',
 'e0104': 'infactus',
 'e0105': 'infactus',
 'e0107': 'Angine',
 'e0108': 'infactus',
 'e0111': 'Angine',
 'e0112': 'Angine',
 'e0113': 'Angine',
 'e0114': 'Angine',
 'e0136': 'Angine',
 'e0139': 'Angine',
 'e0147': 'Angine',
 'e0148': 'Angine',
 'e0154': 'Angine',
 'e0155': 'Angine',
 'e0159': 'infactus',
 'e0161': 'Angine',
 'e0162': 'infactus',
 'e0163': 'infactus',
 'e0166': 'infactus',
 'e0116': 'Angine',
 'e0118': 'Angine',
 'e0119': 'Angine',
 'e0121': 'Angine',
 'e0122': 'Angine',
 'e0123': 'infactus',
 'e0124': 'infactus',
 'e0125': 'infactus',
 'e0126': 'infactus',
 'e0127': 'Angine',
 'e0170': 'Angine'}

In [ ]:
def process_ecg_file(filepath, label):
    try:
        record = wfdb.rdrecord(filepath)
        fs = record.fs
        channel_names = [name.lower() for name in record.sig_name]
        print(channel_names)
        # Find index for V4 (R-peak detection)
        idx_v4 = None
        for ch_name in ['v4']:
            if ch_name in channel_names:
                idx_v4 = channel_names.index(ch_name)
                break
        if idx_v4 is None:
            print(f"❌ Lead V4 not found in: {filepath}")
            return []

        # Find index for D3 / MLIII / III (segment for feature extraction)
        idx_d3 = None
        for ch_name in [ 'mliii', 'iii', 'd3']:
            if ch_name in channel_names:
                idx_d3 = channel_names.index(ch_name)
                break
        if idx_d3 is None:
            print(f"❌ Lead D3/III not found in: {filepath}")
            return []

        lead_v4 = record.p_signal[:, idx_v4]
        lead_d3 = record.p_signal[:, idx_d3]

        # Check for NaNs
        if np.isnan(lead_v4).any() or np.isnan(lead_d3).any():
            print(f"❌ NaNs in signal: {filepath}")
            return []

        # R-peaks from V4
        filtered_v4 = preprocess_ecg(lead_v4, fs)
        normed_v4 =filtered_v4 #(filtered_v4 - np.mean(filtered_v4)) / np.std(filtered_v4)
        r_peaks, _ = pan_tompkins(normed_v4, fs)

        if len(r_peaks) < 2:
            print(f"❌ Not enough R-peaks in: {filepath}")
            return []

        filename = os.path.basename(filepath)  # e.g., e0103
        name_no_ext = filename.replace('.dat', '')
        label_from_csv = labels_map.get(name_no_ext)

        if label_from_csv == 'infactus':
            max_samples = 5 * 60 * fs * 2
        elif label_from_csv is None:
            max_samples = 5 * 60 * fs * 0.5
        else:
            max_samples = 5 * 60 * fs
        feature_list = []

        # Normalize D3 lead
        filtered_d3 = preprocess_ecg(lead_d3, fs)
        normed_d3 = filtered_d3#(filtered_d3 - np.mean(filtered_d3)) / np.std(filtered_d3)

        for i in range(len(r_peaks) - 1):
            start_idx = r_peaks[i]
            end_idx = r_peaks[i + 1]

            if end_idx > max_samples:
                break

            segment_length = end_idx - start_idx

            segment = normed_d3[start_idx:end_idx]  # use D3 signal here
            features = extract_stat_entropy_features(segment)
            feature_list.append((filename, i, features, label))

        return feature_list

    except Exception as e:
        print(f"❌ Error in {filepath}: {e}")
        return []


In [ ]:
import os
import wfdb
import pandas as pd
import numpy as np

# Define paths
path_ischemic = f"{WORK_DIR}/ECG_Data/Ischemic"
path_normal = f"{WORK_DIR}/ECG_Data/Normal"
csv_path = f"{WORK_DIR}/fd3.csv"

# Get .dat files
ischemic_files = [(os.path.join(path_ischemic, f[:-4]), f, 1) for f in os.listdir(path_ischemic) if f.endswith('.dat')]
normal_files = [(os.path.join(path_normal, f[:-4]), f, 0) for f in os.listdir(path_normal) if f.endswith('.dat')]
all_files = ischemic_files + normal_files

# Define feature names
feature_names = [
    'A_max', 'A_min', 'mean', 'std', 'rms',
    'kurtosis', 'skewness', 'sample_entropy',
    'permutation_entropy', 'approximate_entropy'
]

# Function to process each file
def process_and_save(full_path, fname, label):
    try:
        results = process_ecg_file(full_path, label)  # Your custom function
        if results:
            rows = []
            for (filename, segment_index, features, lbl) in results:
                row = {'filename': filename, 'segment_index': segment_index, 'label': lbl}
                row.update({k: v for k, v in zip(feature_names, features)})
                rows.append(row)
            df = pd.DataFrame(rows)
            df.to_csv(csv_path, mode='a', header=not os.path.exists(csv_path), index=False)
            print(f"✅ File processed and saved: {fname}")
        else:
            print(f"⚠️ No data extracted from: {fname}")
    except Exception as e:
        print(f"❌ Error processing {fname}: {e}")

# Loop through all unprocessed files
processed_files = set()
if os.path.exists(csv_path):
    processed_df = pd.read_csv(csv_path)
    processed_files = set(processed_df['filename'].unique())

for full_path, fname, label in all_files:
    if fname[:-4] not in processed_files:
        print(f"🚀 Processing file: {fname}")
        process_and_save(full_path, fname, label)
        # Update processed files to avoid duplicate processing during the loop
        processed_files.add(fname[:-4])

print("✅ All files processed.")


🚀 Processing file: e0159.dat
['mliii', 'v4']
✅ File processed and saved: e0159.dat
🚀 Processing file: e0116.dat
['v4', 'mliii']
✅ File processed and saved: e0116.dat
🚀 Processing file: e0154.dat
['mliii', 'v4']
✅ File processed and saved: e0154.dat
🚀 Processing file: e0161.dat
['v4', 'mliii']
✅ File processed and saved: e0161.dat
🚀 Processing file: e0107.dat
['d3', 'v4']
✅ File processed and saved: e0107.dat
🚀 Processing file: e0104.dat
['mliii', 'v4']
✅ File processed and saved: e0104.dat
🚀 Processing file: e0124.dat
['v4', 'mliii']
✅ File processed and saved: e0124.dat
🚀 Processing file: e0127.dat
['v4', 'mliii']
✅ File processed and saved: e0127.dat
🚀 Processing file: e0163.dat
['mliii', 'v4']
✅ File processed and saved: e0163.dat
🚀 Processing file: e0112.dat
['mliii', 'v4']
✅ File processed and saved: e0112.dat
🚀 Processing file: e0113.dat
['mliii', 'v4']
✅ File processed and saved: e0113.dat
🚀 Processing file: e0166.dat
['v4', 'mliii']
✅ File processed and saved: e0166.dat
🚀 Proce